# Ah Beng Persona Training

Fine-tune `yuhueng/qwen3-4b-singlish-base` into an Ah Beng-style Singlish assistant using local conversation pairs.

This notebook is organized into setup, dataset preparation, training, evaluation, publishing, and reload steps.


## 1. Environment Setup

Install the required packages and version pins for local or Colab execution.


In [ ]:
%%bash
set -e

if env | grep -q '^COLAB_'; then
    XFORMERS=$(python -c 'import re, torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0); print("xformers==" + ("0.0.33.post1" if v == "2.9" else "0.0.32.post2" if v == "2.8" else "0.0.29.post3"))')
    pip install -q --no-deps bitsandbytes accelerate "$XFORMERS" peft trl triton cut_cross_entropy unsloth_zoo
    pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    pip install -q --no-deps unsloth
else
    pip install -q unsloth
fi

pip install -q transformers
pip install -q --no-deps trl


## 2. Imports

Keep the Python imports centralized in one place so the runtime dependencies are easy to audit.


In [ ]:
# Centralized imports
import gc
import json
import time
from pathlib import Path

import torch
from datasets import load_dataset
from huggingface_hub import login
from peft import PeftModel
from transformers import AutoModel
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only


## 3. Training Utilities

Define helper functions for LoRA fine-tuning and response generation.


In [ ]:
def train_model(model_name, train_dataset, config, system_prompt=None):
    """
    Train a model with QLoRA and return training stats.
    Supports continuing training from an existing adapter.
    """

    print(f"Training {model_name}")

    # Track GPU memory before
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    # Load model
    # Unsloth handles loading base model + adapter automatically if an adapter path is provided

    model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="yuhueng/qwen3-4b-singlish-base",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    )

    # Initialize LoRA config
    # If we loaded an adapter, this step ensures Unsloth's training patches are active
    model = FastLanguageModel.get_peft_model(
        model,
        r=config["r"],
        target_modules=config["target_modules"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )


    trainable_params = 0
    for name, param in model.named_parameters():
        if "lora" in name or "modules_to_save" in name:
            param.requires_grad = True
            trainable_params += 1

    if trainable_params == 0:
        print("WARNING: No trainable parameters found! Forcing LoRA gradients...")
        for name, param in model.named_parameters():
            if "lora" in name:
                param.requires_grad = True

    print(f"Verified trainable parameters.")

    # Setup tokenizer
    tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

    # Format dataset
    def formatting_prompts_func(examples):
        convos = examples["conversations"]
        texts = []
        for convo in convos:
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.extend(convo)
            texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
        return {"text": texts}


    formatted_dataset = train_dataset.map(formatting_prompts_func, batched=True)

    # Trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=formatted_dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=config["per_device_train_batch_size"],
            gradient_accumulation_steps=config["gradient_accumulation_steps"],
            warmup_steps=config["warmup_steps"],
            max_steps=config["max_steps"],
            learning_rate=config["learning_rate"],
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            report_to="none",
            output_dir=f"outputs_{model_name}",
        ),
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    # Train
    trainer_stats = trainer.train()

    # Collect metrics
    training_time = time.time() - start_time
    peak_memory = torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024
    final_loss = trainer_stats.metrics.get("train_loss", trainer.state.log_history[-1].get("loss", None))

    # Save adapter
    save_path = f"singlish_adapter_{model_name.replace(' ', '_')}"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    results = {
        "model_name": model_name,
        "training_time_min": round(training_time / 60, 2),
        "peak_vram_gb": round(peak_memory, 2),
        "final_loss": round(final_loss, 4) if final_loss else None,
        "adapter_path": save_path,
    }

    print(f"\n{model_name} Training Complete:")
    print(f"  Time: {results['training_time_min']} min")
    print(f"  Peak VRAM: {results['peak_vram_gb']} GB")
    print(f"  Final Loss: {results['final_loss']}")
    print(f"  Saved to: {save_path}")

    # Cleanup to free VRAM
    del model, trainer
    torch.cuda.empty_cache()

    return results

def generate_response(model, tokenizer, prompt, system_prompt=None, max_new_tokens=128, temperature=0.7, top_p=0.9):
    """
    Generate a response for a given prompt, with an optional system prompt and decoding parameters.
    """
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response.strip()

print("Functions `train_model` and `generate_response` updated.")

## 4. Dataset Loading and Configuration

Load the local Ah Beng persona dataset, validate the expected schema, and define the LoRA training hyperparameters.


In [ ]:
# Save the dataset as `ah_beng_persona.json` either beside this notebook
# or inside the `Persona Training/` folder with records shaped like:
# [{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}]
dataset_candidates = [
    Path("ah_beng_persona.json"),
    Path("Persona Training") / "ah_beng_persona.json",
]
dataset_path = next((path for path in dataset_candidates if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Missing `ah_beng_persona.json`. Save it beside the notebook or in `Persona Training/`."
    )

persona_dataset = load_dataset("json", data_files=str(dataset_path), split="train")

if len(persona_dataset) == 0:
    raise ValueError("The persona dataset is empty.")

sample_record = persona_dataset[0]
if "messages" not in sample_record:
    raise KeyError("Each dataset record must contain a top-level `messages` field.")
if not sample_record["messages"] or not all(
    {"role", "content"} <= set(message) for message in sample_record["messages"]
):
    raise ValueError(
        "Each `messages` entry must be a non-empty list of objects with `role` and `content`."
    )

print(f"Loaded persona dataset from {dataset_path}")

PERSONA_TRAINING_CONFIG = {
    "r": 32,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # All linear layers
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "max_steps": 90,
    "learning_rate": 2e-4,
    "warmup_steps": 5,
}


In [ ]:


persona_dataset = persona_dataset.rename_column("messages", "conversations")

## 5. Inference Utility

Define a small helper for reloading the trained adapter on top of the base model during evaluation.


In [ ]:
def load_trained_model(model_path, adapter_path):
    """
    Load a trained model with its adapter for evaluation.
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=2048,
        load_in_4bit=True,
    )

    model = PeftModel.from_pretrained(model, adapter_path)
    tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

    # Enable inference mode
    FastLanguageModel.for_inference(model)

    return model, tokenizer

print("Function `load_trained_model` redefined.")


## 6. Training Run

### No System Prompt Baseline

Train the Ah Beng adapter using only the conversation pairs from the dataset.

In [ ]:
print("Training Ah Beng persona adapter...")

persona_results_singlish_AhBeng = train_model(
    model_name="4B-AhBeng-on-Singlish_no_system_prompt",
    train_dataset=persona_dataset,
    config=PERSONA_TRAINING_CONFIG,
)

# Save the results of the persona adapter training
with open("training_results_4B_ah_beng_persona_no_system_prompt.json", "w") as f:
    json.dump(persona_results_singlish_AhBeng, f, indent=2)

print("Ah Beng persona adapter training complete and results saved.")


## 7. Persona Sanity Check

Reload the saved adapter and inspect a small prompt set to verify the response style.


In [ ]:
# Force memory cleanup before loading to prevent VRAM errors
gc.collect()
torch.cuda.empty_cache()

print("Loading Ah Beng Adapter...")
model_beng, tokenizer_beng = load_trained_model(
    model_path="yuhueng/qwen3-4b-singlish-base",
    adapter_path="singlish_adapter_4B-AhBeng-on-Singlish_no_system_prompt"
)

# Test prompts
persona_prompts = [
    "why u so stupid knn?",
    "Hi",
    "what's up",
    "How is life",
    "Tell me a story",
    "do you wanna go on a date with me?",
    "Why is the sky blue?",
    "Best place to eat in singapore?"
]

print("\n--- Persona Check ---\n")
for prompt in persona_prompts:
    response = generate_response(
        model_beng,
        tokenizer_beng,
        prompt,
        temperature=0.87, # Specific decoding parameters
        top_p=0.87,
        max_new_tokens=65
    )
    print(f"User: {prompt}")
    print(f"Ah Beng: {response}\n")
